# Бесплатный «мозг» для OpenClaw: проверка ключей + генератор конфига

Ноутбук для Google Colab (GPU **не нужен**, хватит обычного CPU-рантайма).
Ничего не хостит и не держит: работает 30 секунд и выдаёт готовый конфиг.

**Зачем:** OpenClaw нужны две вещи — модель и **tool-calling**. Куча «бесплатных» эндпоинтов отвечает
на вопросы, но не умеет вызывать инструменты, и агент с ними бесполезен. Здесь мы это проверяем
по-настоящему, а не «на глаз».

1. Пункт меню → *Runtime* → *Run all*.
2. В следующей ячейке вставь свои ключи (или положи их в Colab Secrets — тогда ячейку можно не менять).
3. Скопируй вывод последней ячейки в `~/.openclaw/openclaw.json`.


## 1. Что нам выдали

Просто чтобы понимать: 12 ГБ RAM и 2 ядра — это норма для Colab free, GPU тут не нужен.

In [ ]:
import os, platform, shutil, subprocess

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    except Exception as e:
        return f"{e}"

ram_gb = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1)
print(f"Python : {platform.python_version()}")
print(f"RAM    : {ram_gb} GB")
print(f"vCPU   : {os.cpu_count()}")
print(f"Диск   : {shutil.disk_usage('/').free // 10**9} GB свободно")
gpu = sh("nvidia-smi -L")
print(f"GPU    : {gpu if gpu else 'нет (для облачных моделей и не нужен)'}")


## 2. Проверка ключей с настоящим tool-calling

Вставь ключи ниже **или** добавь их в Colab Secrets (иконка ключа слева) под теми же именами —
тогда эта ячейка подхватит их сама. Ключи никуда не отправляются, кроме API самих провайдеров.

Модели в словаре `PROVIDERS` — стартовые. Если провайдер переименовал модель, поправь `model`
по его актуальному списку; это единственное, что тут может устареть.


In [ ]:
import json, os
import requests

try:  # Colab Secrets, если ключи заведены там
    from google.colab import userdata
    for k in ("GEMINI_API_KEY", "GROQ_API_KEY", "OPENROUTER_API_KEY"):
        if not os.environ.get(k):
            try:
                os.environ[k] = userdata.get(k)
            except Exception:
                pass
except ImportError:
    pass

PROVIDERS = {
    "gemini": {
        "base": "https://generativelanguage.googleapis.com/v1beta/openai",
        "model": "gemini-2.5-flash",
        "key_env": "GEMINI_API_KEY",
        "note": "самый щедрый free tier",
    },
    "groq": {
        "base": "https://api.groq.com/openai/v1",
        "model": "llama-3.3-70b-versatile",
        "key_env": "GROQ_API_KEY",
        "note": "очень быстрый",
    },
    "openrouter": {
        "base": "https://openrouter.ai/api/v1",
        "model": "deepseek/deepseek-chat-v3.1:free",
        "key_env": "OPENROUTER_API_KEY",
        "note": "один ключ на десятки моделей",
    },
}

TOOLS = [{
    "type": "function",
    "function": {
        "name": "get_current_time",
        "description": "Возвращает текущее время в указанном городе.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "Название города"}},
            "required": ["city"],
        },
    },
}]

def probe(name, cfg):
    key = os.environ.get(cfg["key_env"], "").strip()
    if not key:
        return {"name": name, "status": "нет ключа", "tools": "-", "ms": "-", "model": cfg["model"]}
    import time
    t0 = time.time()
    try:
        r = requests.post(
            cfg["base"].rstrip("/") + "/chat/completions",
            headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
            json={
                "model": cfg["model"],
                "messages": [{"role": "user", "content": "Который час в Риге? Вызови инструмент."}],
                "tools": TOOLS,
                "tool_choice": "auto",
                "max_tokens": 256,
            },
            timeout=60,
        )
        ms = int((time.time() - t0) * 1000)
        if r.status_code != 200:
            return {"name": name, "status": f"HTTP {r.status_code}", "tools": "-", "ms": ms,
                    "model": cfg["model"], "error": r.text[:160]}
        msg = r.json()["choices"][0]["message"]
        used = bool(msg.get("tool_calls"))
        return {"name": name, "status": "ключ ок", "tools": "ДА" if used else "НЕТ",
                "ms": ms, "model": cfg["model"], "error": "" if used else "модель ответила текстом"}
    except Exception as e:
        return {"name": name, "status": "ошибка", "tools": "-", "ms": "-",
                "model": cfg["model"], "error": str(e)[:160]}

results = [probe(n, c) for n, c in PROVIDERS.items()]

print(f"{'провайдер':<12}{'ключ':<12}{'tools':<7}{'мс':>7}   модель")
print("-" * 72)
for r in results:
    print(f"{r['name']:<12}{r['status']:<12}{r['tools']:<7}{str(r['ms']):>7}   {r['model']}")
    if r.get("error"):
        print(f"{'':<12}↳ {r['error']}")

GOOD = [r["name"] for r in results if r["status"] == "ключ ок" and r["tools"] == "ДА"]
print("\nГодятся для OpenClaw:", ", ".join(GOOD) if GOOD else "ни один — проверь ключи/лимиты")


## 3. Готовый блок конфига

Берём только провайдеров, которые прошли проверку, и собираем fallback-цепочку: первый — основной,
остальные подхватят, когда у первого кончится дневной лимит.


In [ ]:
ORDER = ["gemini", "groq", "openrouter"]
chain = [n for n in ORDER if n in GOOD]

if not chain:
    print("Нечего генерировать: сначала пройди проверку в ячейке выше.")
else:
    refs = [f"{n}/{PROVIDERS[n]['model']}" for n in chain]
    fallback = ", ".join(f'"{r}"' for r in refs[1:])

    print("=== 1. Минимальный вариант (провайдеры уже есть в OpenClaw) ===")
    print(f"""  agents: {{
    defaults: {{
      model: {{
        primary: "{refs[0]}",
        fallback: [{fallback}],
      }},
    }},
  }},""")
    print("\nСверь id моделей со списком: openclaw models list\n")

    print("=== 2. Только если нужной модели НЕТ в каталоге OpenClaw ===")
    print("(переопределяем провайдера вручную; для bundled-провайдеров это не нужно)")
    providers_block = ",\n".join(
        f"""      {n}: {{
        api: "openai-completions",
        baseUrl: "{PROVIDERS[n]['base']}",
        apiKey: "${{{PROVIDERS[n]['key_env']}}}",
        models: [{{ id: "{PROVIDERS[n]['model']}", name: "{PROVIDERS[n]['model']}" }}],
      }}"""
        for n in chain
    )
    print(f"""  models: {{
    mode: "merge",
    providers: {{
{providers_block}
    }},
  }},""")

    print("\nДальше: openclaw doctor --fix && openclaw models status --probe")


## 4. Эксперимент: своя модель прямо в Colab (не для дежурства)

**Прочитай предупреждения, прежде чем жать Run.**

- Рантайм живёт максимум 12 ч, при простое его убивают, диск стирается. Это тест, а не хостинг.
- На free-рантайме обычно нет GPU → 7–8B модель отвечает секунды на токен. Для агентского цикла
  (десятки вызовов инструментов) это мучение. С T4 в Colab Pro — уже терпимо.
- `cloudflared` quick tunnel делает твой эндпоинт **открытым в интернет без пароля**. Кто узнал URL —
  тот гоняет твою модель и жжёт твою квоту. Для эксперимента на 20 минут — ок, на ночь — нет.
- URL меняется при каждом перезапуске → потом править `baseUrl` в конфиге руками.

Если всё это устраивает — запускай. В OpenClaw подключай как `api: "ollama"`, `baseUrl` **без** `/v1`.


In [ ]:
# 1) ставим Ollama и тянем компактную модель (~5 ГБ, влезает в RAM free-рантайма)
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/install.log 2>&1 && echo "ollama установлена"
!nohup ollama serve > /tmp/ollama.log 2>&1 &
!sleep 5 && ollama pull qwen3:8b && ollama list

# 2) пробуем локально (на CPU это будет медленно — это нормально)
!curl -s http://127.0.0.1:11434/api/tags | head -c 300


In [ ]:
# 3) публичный туннель (НЕ оставляй включённым без присмотра)
!curl -fsSL -o /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared
!nohup /tmp/cloudflared tunnel --url http://127.0.0.1:11434 --no-autoupdate > /tmp/tunnel.log 2>&1 &
!sleep 10
!grep -oE 'https://[a-z0-9-]+\.trycloudflare\.com' /tmp/tunnel.log | head -1

print("""
Скопируй URL и подставь в openclaw.json:

  models: { providers: { colab: {
    api: "ollama",              // без /v1 !
    baseUrl: "<URL из вывода выше>",
    apiKey: "colab-local",
    timeoutSeconds: 300,
    models: [{ id: "qwen3:8b", name: "Qwen3 8B (Colab)" }],
  }}}
  agents: { defaults: { model: { primary: "colab/qwen3:8b" } } }

Потом: openclaw models status --probe
Остановить: Runtime -> Disconnect and delete runtime
""")


## 5. Что дальше

- Конфиг целиком (с hardened-гейтвеем и Telegram): `openclaw/openclaw.json` в репозитории.
- Про хостинг при 4 ГБ ОЗУ и почему Colab не дежурный сервер: `docs/hosting-4gb-colab.md`.
- Бэкап состояния в Google Drive: `scripts/backup-openclaw-to-drive.sh`.
